In [ ]:
!pip install -q -U transformers datasets bitsandbytes  trl peft  huggingface_hub

In [1]:
from huggingface_hub import login,whoami
login()
print(whoami())

{'type': 'user', 'id': '69d4675c26493c5725f3e1ea', 'name': 'Chufeng-Jiang', 'fullname': 'Chufeng Jiang', 'email': 'jiangchufengjcf@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1777593600, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/69d4675c26493c5725f3e1ea/pY8UdnpJSc0loAQ9Ac_qi.jpeg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'test', 'role': 'write', 'createdAt': '2026-04-07T02:12:24.103Z'}}}


In [ ]:
# Prompt:       [PAD, PAD, 1, 2]       (left-pad)
# Chosen:       [3, 4, 5, PAD]         (right-pad)
# Rejected:     [6, PAD, PAD, PAD]     (right-pad)

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch


name_name = "Qwen/Qwen2.5-1.5B-Instruct"

config_8bit = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoModelForCausalLM.from_pretrained(
                                                  name_name,
                                                  quantization_config=config_8bit,
                                                  trust_remote_code=True,
                                                  device_map="auto",
                                                  )

tokenizer = AutoTokenizer.from_pretrained(name_name, trust_remote_code=True)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
from datasets import load_dataset

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset",split="train[:300]")
dataset

README.md: 0.00B [00:00, ?B/s]

data.json:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10884 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [4]:
dataset[0]

{'prompt': 'Oh, I just saw the best meme - have you seen it?',
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [5]:
example_text = [{'role':'assistant','content':dataset[0]['chosen']}]
example_text

[{'role': 'assistant',
  'content': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣"}]

In [6]:
tokenizer.apply_chat_template(example_text,tokenize=False)

"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n"

In [7]:
def apply_chat_temp(example):
    example['prompt'] =[{'role':'user','content':example['prompt']}]
    return example

new_dataset = dataset.map(apply_chat_temp)
new_dataset[0]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

{'prompt': [{'content': 'Oh, I just saw the best meme - have you seen it?',
   'role': 'user'}],
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [8]:
new_templete = """<|im_start|>assistant\n{model_answer}<|im_end|>\n"""


def format_dataset(example):

    example['prompt'] = tokenizer.apply_chat_template(example['prompt'],tokenize=False)
    example['chosen'] = new_templete.format(model_answer =example['chosen'])
    example['rejected'] = new_templete.format(model_answer =example['rejected'])

    return example


formatted_dataset = new_dataset.map(format_dataset)
formatted_dataset[0]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nOh, I just saw the best meme - have you seen it?<|im_end|>\n',
 'chosen': "<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n",
 'rejected': "<|im_start|>assistant\nI'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?<|im_end|>\n"}

In [9]:
model_8bit

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear8bitLt(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear8bitLt(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear8bitLt(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear8bitLt(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Q

In [10]:
from peft import LoraConfig, get_peft_model

lora_config= LoraConfig(
                        r = 32,
                        lora_alpha = 32,
                        target_modules =["q_proj","k_proj","v_proj","o_proj",
                  "gate_proj","up_proj","down_proj"],
                        lora_dropout = 0.05,
                        bias = "none",
                        task_type = "CAUSAL_LM"
)

lora_model = get_peft_model(model_8bit,lora_config)
lora_model.print_trainable_parameters()

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [11]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [12]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
                      output_dir = "./results",
                      per_device_train_batch_size = 2,
                      max_steps = 40,
                      gradient_accumulation_steps = 4,
                      learning_rate = 3e-5,
                      lr_scheduler_type ="cosine",
                      optim = "adamw_torch",
                      logging_steps = 10,
                      beta = 0.01,
                      report_to = "none"
)

In [13]:
trainer = DPOTrainer(
                     model = lora_model,
                    # If no `ref_model` is provided, the trainer creates a copy of the current model (without LoRA adapters).,
                     train_dataset = formatted_dataset,
                    #  tokenizer = tokenizer,  old one
                    processing_class = tokenizer, # new one
                     args = dpo_config
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/home/chufeng/SoftwareInstalled/anaconda3/envs/LLM/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
10,0.619713
20,0.322605
30,0.080632
40,0.041296


TrainOutput(global_step=40, training_loss=0.26606164053082465, metrics={'train_runtime': 251.5056, 'train_samples_per_second': 1.272, 'train_steps_per_second': 0.159, 'total_flos': 1431317837107200.0, 'train_loss': 0.26606164053082465})

In [14]:
lora_path = "./lora_adapters"
lora_model.save_pretrained(lora_path)

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

lora_path = "./lora_adapters"
name_name = "Qwen/Qwen2.5-1.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
                                                  name_name,
                                                  trust_remote_code=True,
                                                  device_map="auto",
                                                  )

tokenizer = AutoTokenizer.from_pretrained(name_name, trust_remote_code=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [20]:
from peft import PeftModel

final_model = PeftModel.from_pretrained(base_model,lora_path)
final_model

/home/chufeng/SoftwareInstalled/anaconda3/envs/LLM/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [21]:
final_model = final_model.merge_and_unload()
final_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [22]:
final_model.push_to_hub("Chufeng-Jiang/Qwen2.5-1.5B-HumanPreference-DPO")
tokenizer.push_to_hub("Chufeng-Jiang/Qwen2.5-1.5B-HumanPreference-DPO")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

HfHubHTTPError: Server error '504 Gateway Time-out' for url 'https://huggingface.co/api/models/Chufeng-Jiang/Qwen2.5-1.5B-HumanPreference-DPO/commit/main' (Amz CF ID: T3slfARhntl8nsynoSjkuZqSoDOCQTHKhya9ikopdvxsCfBtvwwsTA==)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/504